
# STRATUM — Machine Learning Dataset Preparation

## Phase 2 — Step 2.7

### Objective

Prepare reproducible training, validation, and test
partitions for semiconductor virtual metrology.

### Prediction Targets

1. mean_thickness
2. mean_resistance

### Evaluation Strategy

- Chronological data partitioning
- Production-lot independence
- Separate model-development and test partitions
- Training-only preprocessing
- Final test set reserved for Phase 3 evaluation

### Principles

- No random row shuffling
- No post-metrology inputs
- No preprocessing fitted to validation or test data
- No final test-set evaluation in this notebook
- No model training at this stage
- Original dataset remains unchanged

In [1]:

# STEP 2.7.3 - Import libraries and local modules

from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

current_dir = Path.cwd()

if (current_dir / "ml" / "preprocessing.py").exists():
    project_root = current_dir
elif (
    current_dir.parent / "ml" / "preprocessing.py"
).exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Cannot locate STRATUM project root."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ml.preprocessing import (
    get_feature_columns,
    build_preprocessor,
    TARGET_COLUMNS,
)

from ml.data_split import chronological_lot_split

print("STRATUM ML preparation modules imported!")

STRATUM ML preparation modules imported!


## 1. Original Dataset Validation

Load the original dataset and verify its schema
before splitting or applying preprocessing.

In [2]:

# STEP 2.7.4 - Load the original dataset

csv_path = (
    project_root
    / "data"
    / "raw"
    / "apc_spc_AlCu_inputs_outputs.csv"
)

df = pd.read_csv(
    csv_path,
    sep=";",
    skiprows=1,
    low_memory=False,
)

assert df.shape == (5208, 178)

assert df["wafer_id"].is_unique

assert df[TARGET_COLUMNS].notna().all().all()

apc_features, feature_columns = get_feature_columns(df)

print("Total records:", len(df))
print("APC features:", len(apc_features))
print("Candidate input features:", len(feature_columns))
print("Prediction targets:", TARGET_COLUMNS)

print("\nDataset validation passed!")

Total records: 5208
APC features: 110
Candidate input features: 111
Prediction targets: ['mean_thickness', 'mean_resistance']

Dataset validation passed!


## 2. Chronological and Lot-Aware Splitting

Create training, validation, and test partitions.

Reserve approximately 20% of the latest records for
final evaluation.

Use the earlier records for training and validation.

Verify that production lots do not overlap.

In [3]:

# STEP 2.7.5 - Create data partitions

train_df, val_df, test_df, split_info = (
    chronological_lot_split(
        df,
        test_fraction=0.20,
        validation_fraction=0.20,
    )
)

print("STRATUM dataset partitions\n")

for key, value in split_info.items():
    print(f"{key}: {value}")

STRATUM dataset partitions

total_records: 5208
train_records: 3331
validation_records: 834
test_records: 1042
purged_development_records: 0
purged_training_records: 1
train_start: 2021-07-31 06:51:54
train_end: 2023-12-26 05:13:18
validation_start: 2023-12-26 08:01:03
validation_end: 2024-07-31 13:58:35
test_start: 2024-07-31 17:05:11
test_end: 2025-05-31 17:04:14


## 3. Split Integrity Verification

Confirm that all partitions are chronologically ordered.

Verify that production lots and wafer IDs are unique
across training, validation, and testing.

In [4]:

# STEP 2.7.6 - Verify the split

partitions = {
    "Training": train_df,
    "Validation": val_df,
    "Testing": test_df,
}

summary_rows = []

for name, part in partitions.items():
    summary_rows.append({
        "Partition": name,
        "Records": len(part),
        "Unique Lots": part["lot"].nunique(),
        "Equipment IDs": part["equipment"].nunique(),
        "First Run": part["runstart"].min(),
        "Last Run": part["runstart"].max(),
    })

split_summary = pd.DataFrame(summary_rows)

display(split_summary)

# Lot overlap verification
train_lots = set(train_df["lot"])
val_lots = set(val_df["lot"])
test_lots = set(test_df["lot"])

assert train_lots.isdisjoint(val_lots)
assert train_lots.isdisjoint(test_lots)
assert val_lots.isdisjoint(test_lots)

# Timestamp verification
assert (
    train_df["runstart"].max()
    <= val_df["runstart"].min()
)

assert (
    val_df["runstart"].max()
    <= test_df["runstart"].min()
)

# Verify all wafer identifiers remain unique
combined_ids = pd.concat([
    train_df["wafer_id"],
    val_df["wafer_id"],
    test_df["wafer_id"],
])

assert combined_ids.is_unique

print("Chronological ordering: PASSED")
print("Production-lot independence: PASSED")
print("Wafer ID uniqueness: PASSED")

,Partition,Records,Unique Lots,Equipment IDs,First Run,Last Run
0,Training,3331,1874,16,2021-07-31 06:51:54,2023-12-26 05:13:18
1,Validation,834,487,16,2023-12-26 08:01:03,2024-07-31 13:58:35
2,Testing,1042,590,13,2024-07-31 17:05:11,2025-05-31 17:04:14


Chronological ordering: PASSED
Production-lot independence: PASSED
Wafer ID uniqueness: PASSED


## 4. Equipment Coverage

Examine equipment identifiers shared between
training, validation, and test data.

Identify equipment categories absent from training
but present in later partitions.

In [5]:

# STEP 2.7.7 - Equipment coverage

train_equipment = set(train_df["equipment"])
val_equipment = set(val_df["equipment"])
test_equipment = set(test_df["equipment"])

print("Training equipment:", len(train_equipment))
print("Validation equipment:", len(val_equipment))
print("Testing equipment:", len(test_equipment))

print("\nEquipment unseen in training but in validation:")
print(sorted(val_equipment - train_equipment))

print("\nEquipment unseen in training but in testing:")
print(sorted(test_equipment - train_equipment))

print("\nEquipment present in training but absent in testing:")
print(sorted(train_equipment - test_equipment))

Training equipment: 16
Validation equipment: 16
Testing equipment: 13

Equipment unseen in training but in validation:
[]

Equipment unseen in training but in testing:
[]

Equipment present in training but absent in testing:
['equipment_11', 'equipment_3', 'equipment_7']


## 5. Model Inputs and Targets

Separate permitted manufacturing inputs from
post-process measurement targets.

Prepare both thickness and sheet-resistance
targets without combining them with model inputs.

In [6]:

# STEP 2.7.8 - Separate X and y

X_train = train_df[feature_columns].copy()
X_val = val_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df[TARGET_COLUMNS].copy()
y_val = val_df[TARGET_COLUMNS].copy()

# Reserve the test targets for final Phase 3 evaluation.
# Do not inspect them for model selection here.

assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)

assert set(feature_columns).isdisjoint(TARGET_COLUMNS)

print("Training input shape:", X_train.shape)
print("Validation input shape:", X_val.shape)
print("Testing input shape:", X_test.shape)

print("\nTraining targets:", y_train.columns.tolist())

print("\nInput/target separation passed!")

Training input shape: (3331, 111)
Validation input shape: (834, 111)
Testing input shape: (1042, 111)

Training targets: ['mean_thickness', 'mean_resistance']

Input/target separation passed!


## 6. Training-Only Preprocessing

Fit the preprocessing transformer using X_train only.

Transform validation data using the fitted transformer.

Do not fit or adapt preprocessing using validation
or test data.

Do not train an ML estimator in this notebook.

In [7]:

# STEP 2.7.9 - Fit training-only preprocessing

preprocessor = build_preprocessor(
    numeric_features=apc_features
)

# Learn imputation values and categories from training only
X_train_prepared = preprocessor.fit_transform(X_train)

# Apply those learned rules to validation data
X_val_prepared = preprocessor.transform(X_val)

print("Original training shape:", X_train.shape)
print("Prepared training shape:", X_train_prepared.shape)

print("\nOriginal validation shape:", X_val.shape)
print("Prepared validation shape:", X_val_prepared.shape)

assert (
    X_train_prepared.shape[1]
    == X_val_prepared.shape[1]
)

print("\nTraining-only preprocessing passed!")

Original training shape: (3331, 111)
Prepared training shape: (3331, 173)

Original validation shape: (834, 111)
Prepared validation shape: (834, 173)

Training-only preprocessing passed!


## 7. Preprocessing Integrity Checks

Verify that the prepared matrices:

- Have matching feature dimensions.
- Contain finite numerical values.
- Contain no unresolved missing values.
- Have consistent transformed feature names.

In [8]:

# STEP 2.7.10 - Verify prepared features

feature_names = preprocessor.get_feature_names_out()

assert X_train_prepared.shape[1] == len(feature_names)
assert X_val_prepared.shape[1] == len(feature_names)

assert np.isfinite(X_train_prepared).all()
assert np.isfinite(X_val_prepared).all()

print("Prepared feature count:", len(feature_names))
print("Training NaN count:", np.isnan(X_train_prepared).sum())
print("Validation NaN count:", np.isnan(X_val_prepared).sum())

print("\nFirst 15 transformed feature names:")
for name in feature_names[:15]:
    print(name)

print("\nPreprocessing integrity checks passed!")

Prepared feature count: 173
Training NaN count: 0
Validation NaN count: 0

First 15 transformed feature names:
numeric__AMPS_TARGET_20_90_DELTA_DEP1
numeric__AMPS_TARGET_20_90_DEP1_M
numeric__AMPS_TARGET_20_90_DEP1_SPS
numeric__AMPS_TARGET_20_90_DEP1_SPS_M
numeric__AMPS_TARGET_20_90_DEP1_SPS_X
numeric__AMPS_TARGET_20_90_DEP1_X
numeric__AMPS_TARGET_20_95_DELTA_DEP1
numeric__AMPS_TARGET_20_95_DEP1_M
numeric__AMPS_TARGET_20_95_DEP1_SPS
numeric__AMPS_TARGET_20_95_DEP1_SPS_M
numeric__AMPS_TARGET_20_95_DEP1_SPS_X
numeric__AMPS_TARGET_20_95_DEP1_X
numeric__AMPS_TARGET_DEP1_A
numeric__AMPS_TARGET_DEP1_A_MR
numeric__AMPS_TARGET_DEP1_S

Preprocessing integrity checks passed!


## 8. Unseen Equipment Handling

Create a synthetic example with an equipment identifier
not observed during training.

Verify that the fitted preprocessor handles this
category without raising an exception.

This is a software robustness test, not evidence
of predictive accuracy on new equipment.

In [9]:

# STEP 2.7.11 - Test unseen equipment

synthetic_input = X_train.iloc[[0]].copy()

synthetic_input["equipment"] = (
    "equipment_unseen_demo"
)

synthetic_prepared = preprocessor.transform(
    synthetic_input
)

assert synthetic_prepared.shape[0] == 1

assert (
    synthetic_prepared.shape[1]
    == X_train_prepared.shape[1]
)

assert np.isfinite(synthetic_prepared).all()

print("Unseen equipment handled successfully!")
print("Output shape:", synthetic_prepared.shape)

Unseen equipment handled successfully!
Output shape: (1, 173)


## 9. Reproducibility Manifest

Save the splitting configuration and partition summary.

The manifest documents the reproducible evaluation
protocol but does not contain original wafer measurements.

In [10]:

# STEP 2.7.12 - Save split configuration

manifest = {
    "project": "STRATUM",
    "dataset_version": "Infineon AlCu v2",
    "source_file": "apc_spc_AlCu_inputs_outputs.csv",
    "method": "chronological_lot_aware",
    "test_fraction": 0.20,
    "validation_fraction_of_development": 0.20,
    "ordering": ["runstart", "wafer_id"],
    "boundary_policy": "purge_earlier_overlapping_lots",
    "targets": TARGET_COLUMNS,
    "numeric_apc_feature_count": len(apc_features),
    "categorical_features": ["equipment"],
    "split_summary": split_info,
}

manifest_path = (
    project_root / "docs" / "split_manifest.json"
)

manifest_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=4,
    )

print("Split manifest saved:")
print("docs/split_manifest.json")

Split manifest saved:
docs/split_manifest.json



## 10. Dataset Preparation Findings

### Dataset
- Original wafer records: 5,208
- Original columns: 178
- Numeric APC inputs: 110
- Optional categorical input: equipment
- Regression targets: mean_thickness and mean_resistance

### Splitting Method
- Chronological ordering based on runstart.
- Approximately 80% development and 20% testing.
- Development further divided into training and validation.
- Lot overlap prevented at partition boundaries.
- Excluded boundary records documented when applicable.

### Preprocessing
- Numeric median imputation fitted on training data.
- Numeric missingness indicators.
- One-hot equipment encoding.
- Unknown equipment categories handled.
- No post-process metrology inputs.
- No test data used for preprocessing fitting.

### Verification
- Chronological ordering checked.
- Production-lot independence checked.
- Wafer identifier uniqueness checked.
- Transformed feature dimensions verified.
- Missing and non-finite transformed values checked.
- Synthetic unseen-equipment input tested.

### Evaluation Policy
- Training data: model parameter learning.
- Validation data: model comparison and tuning.
- Testing data: final held-out evaluation.
- Final test metrics will be calculated during Phase 3.
- Model selection must not use the test targets.

### Limitations
- Lot separation does not remove every manufacturing dependency.
- Equipment distributions can change over time.
- Unknown equipment categories may reduce predictive accuracy.
- Median imputation is an initial baseline.
- Feature availability at prediction time requires verification.
- No regression model has been trained yet.

### Conclusion
STRATUM now has a reproducible, chronological and
lot-aware ML preparation workflow.

The next phase will train and independently evaluate
virtual metrology regression models.